# 🗽 NYC Taxi Data Ingestion - Bronze Layer
> **Goal:** Pull raw parquet data from the NYC TLC public repository and land it in the Lakehouse. 🚕

### 🛠️ Engineering Steps:
1. **Define** the source URL for the parquet file.
2. **Download** the file to the Lakehouse 'Files' section.
3. **Read** the data using PySpark.
4. **Enrich** with a `load_timestamp` and `source_file`.
5. **Write** to the Bronze Delta table using **Append** mode (Full Ingestion).

In [1]:
import requests
import os
from pyspark.sql.functions import current_timestamp, lit

# 1. Configuration & Directory Setup
# We'll pull Jan and Feb 2024 to demonstrate incremental processing later
months = ["2024-01", "2024-02"]
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{}.parquet"
raw_folder_path = "/lakehouse/default/Files/raw_data"

# Create the directory if it doesn't exist to avoid FileNotFoundError 🛠️
if not os.path.exists(raw_folder_path):
    os.makedirs(raw_folder_path)
    print(f"📁 Created directory: {raw_folder_path}")

# 2. Loop through months to Download and Load
for month in months:
    file_name = f"yellow_tripdata_{month}.parquet"
    url = base_url.format(month)
    local_path = f"{raw_folder_path}/{file_name}"
    
    print(f"🚀 Processing: {file_name}")
    
    # Download File
    response = requests.get(url)
    if response.status_code == 200:
        with open(local_path, "wb") as f:
            f.write(response.content)
        print(f"✅ Downloaded to Files: {file_name}")
        
        # 3. Read and Add Metadata
        df_raw = spark.read.parquet(f"Files/raw_data/{file_name}")
        df_bronze = df_raw.withColumn("load_timestamp", current_timestamp()) \
                          .withColumn("source_file", lit(file_name))
        
        # 4. Append to Bronze Table
        # Using 'append' ensures we keep a history of every file processed
        df_bronze.write.format("delta").mode("append").saveAsTable("bronze_nyc_taxi")
        print(f"📦 {file_name} appended to bronze_nyc_taxi. Rows: {df_bronze.count()}")
    else:
        print(f"❌ Failed to download {file_name}. Status: {response.status_code}")

print("\n✨ Bronze Layer Ingestion Complete!")

StatementMeta(, e1837dfb-601c-4d51-aace-e3a60545515d, 3, Finished, Available, Finished, False)

📁 Created directory: /lakehouse/default/Files/raw_data
🚀 Processing: yellow_tripdata_2024-01.parquet
✅ Downloaded to Files: yellow_tripdata_2024-01.parquet
📦 yellow_tripdata_2024-01.parquet appended to bronze_nyc_taxi. Rows: 2964624
🚀 Processing: yellow_tripdata_2024-02.parquet
✅ Downloaded to Files: yellow_tripdata_2024-02.parquet
📦 yellow_tripdata_2024-02.parquet appended to bronze_nyc_taxi. Rows: 3007526

✨ Bronze Layer Ingestion Complete!
